In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/thien1997nus@gmail.com/fmcg_databricks/src/1_setup/utilities
# import name from utilities for bronze_schema etc

In [0]:
# create widget so that can get data setup from it
# to specify environment when dev/prod, other datasources
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
## move files . use when wrong setup only
dbutils.fs.mv("/Volumes/fmcg/default/fmcg/1_parent_company/full_load/dim_customers.csv", "/Volumes/fmcg/default/fmcg/1_parent_company/full_load/customers/dim_customers.csv")

In [0]:
# path data volumne
base_path = f'/Volumes/fmcg/default/{catalog}/2_child_company/full_load/{data_source}/*.csv'
print(base_path)





In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select( "*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(10))

In [0]:
df.write\
    .format("delta")\
    .option( "delta.enableChangeDataFeed", "true") \
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")